In [3]:
import pandas as pd
import random

In [2]:
import os

print(os.getcwd())

/data2/ruichenzheng/psychdepth_v2/pipeline


In [8]:
folder_path = 'Qwen2.5-32B-Instruct_OpenMath-Nemotron-32B_eval_math500'
base_model = folder_path.split('_')[0]
reasoning_model = folder_path.split('_')[1]
premises = pd.read_csv("/data2/ruichenzheng/psychdepth_v2/data/premises.csv")

In [9]:
# Load stories from both models
df_base = pd.read_csv(f"results/{folder_path}/{base_model}_stories.csv")
df_reasoning = pd.read_csv(f"results/{folder_path}/{reasoning_model}_stories.csv")

# Merge on premise_id
merged = pd.merge(df_base, df_reasoning, on="premise_id", suffixes=('_base', '_reasoning'))

# Shuffle base/reasoning story order per row
shuffled_data = []
for _, row in merged.iterrows():
    if random.random() < 0.5:
        story_1 = row['text_base']
        story_2 = row['text_reasoning']
        model_1 = base_model
        model_2 = reasoning_model
    else:
        story_1 = row['text_reasoning']
        story_2 = row['text_base']
        model_1 = reasoning_model
        model_2 = base_model
    
    shuffled_data.append({
        'premise_id': row['premise_id'],
        'story_id_base': row['story_id_base'],
        'story_id_reasoning': row['story_id_reasoning'],
        'story_1': story_1,
        'story_2': story_2,
        'model_1': model_1,
        'model_2': model_2
    })

# Create DataFrame
shuffled_df = pd.DataFrame(shuffled_data)

# Save to CSV
shuffled_df.to_csv(f"{base_model}_{reasoning_model}_pairwise_stories.csv", index=False)

In [12]:
import os

csv_files = [f for f in os.listdir() if f.endswith('.csv')]
print(csv_files)


['Llama-3.1-8B-Instruct_OpenMath2-Llama3.1-8B_pairwise_stories.csv', 'phi-4_Phi-4-reasoning_pairwise_stories.csv', 'Qwen2.5-32B-Instruct_Deductive-Reasoning-Qwen-32B_pairwise_stories.csv', 'Phi-4-mini-instruct_Phi-4-mini-reasoning_pairwise_stories.csv', 'Qwen2.5-32B-Instruct_OpenMath-Nemotron-32B_pairwise_stories.csv', 'Llama-3.1-8B-Instruct_Llama-3.1-Nemotron-Nano-8B-v1_pairwise_stories.csv']


In [13]:
import pandas as pd
import glob

# Find all CSV files in the current directory
csv_files = glob.glob("*.csv")

# Read and concatenate them into one DataFrame
df_all = pd.concat((pd.read_csv(f) for f in csv_files), ignore_index=True)
df_merged = pd.merge(df_all, premises, on="premise_id", how="left")
# Save to a single CSV
df_merged.to_csv("merged_all.csv", index=False)

print(f"Merged {len(csv_files)} files into 'merged_all.csv'")


Merged 6 files into 'merged_all.csv'


In [15]:
df_sampled = df_merged.sample(n=50, random_state=42).reset_index(drop=True)

# Save to CSV
df_sampled.to_csv("sampled_50_random_pairs.csv", index=True)
print("Saved 50 random pairs to sampled_50_random_stories.csv")

Saved 50 random pairs to sampled_50_random_stories.csv
